This notebook creates the reusable gold-layer category BI view `adwm_wh.gold.vw_bi_sales_by_category` for downstream merchandise reporting.

Scope:

* Create or replace the category BI view in `adwm_wh.gold`
* Reuse `adwm_wh.gold.vw_bi_factsales_base` as the source layer
* Summarize sales performance by category and subcategory
* Run lightweight validation for top category results

Business grain:

* One row per category and subcategory

Note:

* If category attributes remain unresolved in the source dimensions, the view still gets created and validation will surface `Unknown` groupings.

In [0]:
%sql
CREATE OR REPLACE VIEW adwm_wh.gold.vw_bi_sales_by_category AS
SELECT
  CategoryName,
  SubcategoryName,
  COUNT(*) AS SalesLineCount,
  COUNT(DISTINCT SalesOrderNumber) AS SalesOrderCount,
  COUNT(DISTINCT ProductKey) AS ProductCount,
  SUM(OrderQuantity) AS TotalOrderQuantity,
  CAST(SUM(SalesAmount) AS DECIMAL(19,4)) AS TotalSalesAmount,
  CAST(SUM(TotalCost) AS DECIMAL(19,4)) AS TotalCost,
  CAST(SUM(DiscountAmount) AS DECIMAL(19,4)) AS TotalDiscountAmount,
  CAST(SUM(GrossMargin) AS DECIMAL(19,4)) AS GrossMargin
FROM adwm_wh.gold.vw_bi_factsales_base
GROUP BY CategoryName, SubcategoryName;

In [0]:
%sql
SELECT
  CategoryName,
  SubcategoryName,
  SalesLineCount,
  SalesOrderCount,
  ProductCount,
  TotalOrderQuantity,
  CAST(TotalSalesAmount AS DECIMAL(19,2)) AS TotalSalesAmount,
  CAST(GrossMargin AS DECIMAL(19,2)) AS GrossMargin,
  CAST(TotalDiscountAmount AS DECIMAL(19,2)) AS TotalDiscountAmount
FROM adwm_wh.gold.vw_bi_sales_by_category
ORDER BY TotalSalesAmount DESC, CategoryName, SubcategoryName
LIMIT 10;